# 09 QGIS Export

**Objective:** export the neighbourhood-level results of notebooks 04-07 to a single GeoPackage (`outputs/qgis/airbnb_nyc_results.gpkg`) for mapping in QGIS.

**Layers:** `nta_results` (197 residential NTAs, one row per NTA, results for 2022-2025 in separate columns), `nta_all_context` (all 262 NTAs) and `boroughs`. Coordinate system: EPSG:2263. No listing-level data is exported. Median rates are shown only for NTA-years with at least 5 listings, as in the analysis.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, PROCESSED, TAB = PROJECT_ROOT / "data" / "raw", PROJECT_ROOT / "data" / "processed", PROJECT_ROOT / "outputs" / "tables"
QGIS_DIR = PROJECT_ROOT / "outputs" / "qgis"
QGIS_DIR.mkdir(parents=True, exist_ok=True)
GPKG = QGIS_DIR / "airbnb_nyc_results.gpkg"

YEARS, MIN_LISTINGS = [2022, 2023, 2024, 2025], 5

nta = gpd.read_file(RAW / "nta2020.geojson")[["nta2020", "ntaname", "boroname", "ntatype", "geometry"]].to_crs(2263)
nty = pd.read_parquet(PROCESSED / "nta_year_medians.parquet")
lisa = pd.read_parquet(PROCESSED / "lisa_results.parquet")
status = pd.read_csv(TAB / "07_cluster_status_by_year.csv")
chg = pd.read_csv(TAB / "04_pct_change_2022_2025.csv")
cat = pd.read_csv(TAB / "07_cluster_transition_categories.csv")
print("NTAs:", len(nta), "| NTA-year medians:", len(nty), "| LISA rows:", len(lisa), "| status rows:", len(status))

NTAs: 262 | NTA-year medians: 730 | LISA rows: 457 | status rows: 197


In [2]:
res = nta[nta["ntatype"] == "0"][["nta2020", "ntaname", "boroname", "geometry"]].reset_index(drop=True)

for y in YEARS:
    a = nty[nty["year"] == y].set_index("nta2020")
    n = res["nta2020"].map(a["n_listings"]).fillna(0).astype(int)          # no listings = 0
    res[f"n_listings_{y}"] = n
    res[f"rate_{y}"] = res["nta2020"].map(a["median_rate"]).where(n >= MIN_LISTINGS).round(1)   # only if enough listings
    l = lisa[lisa["year"] == y].set_index("nta2020")
    res[f"lisa_fdr_{y}"] = res["nta2020"].map(l["cluster_fdr"]).fillna("Not analysed")
    res[f"lisa_p05_{y}"] = res["nta2020"].map(l["cluster_p05"]).fillna("Not analysed")
    res[f"lisa_pval_{y}"] = res["nta2020"].map(l["p_sim"]).round(3)
    res[f"status_{y}"] = res["nta2020"].map(status.set_index("nta2020")[str(y)])

res["pct_change_22_25"] = res["nta2020"].map(chg.set_index("nta2020")["pct_change"]).round(1)
res["transition"] = res["nta2020"].map(cat.set_index("nta2020")["category"]).fillna("No significant cluster or not analysed")

print(res.drop(columns="geometry").shape, "| rows with a 2022 rate:", int(res["rate_2022"].notna().sum()),
      "| 2025:", int(res["rate_2025"].notna().sum()))
print(res["lisa_fdr_2022"].value_counts().to_string())
print(res["transition"].value_counts().to_string())

(197, 29) | rows with a 2022 rate: 129 | 2025: 98
lisa_fdr_2022
Not significant    108
Not analysed        73
High-High           14
Low-Low              2
transition
No significant cluster or not analysed      176
High-High persisted                           7
High-High lost (not significant in 2025)      5
Cluster in 2023-2024 only                     3
Low-Low persisted                             2
Low-Low emerged by 2025                       2
High-High in 2022 only                        1
High-High became Low-High outlier             1


In [3]:
if GPKG.exists():
    GPKG.unlink()                                          # start from a clean file

boro = nta.dissolve(by="boroname").reset_index()[["boroname", "geometry"]]
nta.to_file(GPKG, layer="nta_all_context", driver="GPKG")
res.to_file(GPKG, layer="nta_results", driver="GPKG")
boro.to_file(GPKG, layer="boroughs", driver="GPKG")

print(gpd.list_layers(GPKG).to_string())
chk = gpd.read_file(GPKG, layer="nta_results")
print("\nRead back:", chk.shape, "| CRS:", chk.crs.to_string(), "| file size:", round(GPKG.stat().st_size / 1e6, 2), "MB")
print("Columns:", [c for c in chk.columns if c != "geometry"])

              name geometry_type
0  nta_all_context  MultiPolygon
1      nta_results  MultiPolygon
2         boroughs  MultiPolygon

Read back: (197, 30) | CRS: EPSG:2263 | file size: 5.03 MB
Columns: ['nta2020', 'ntaname', 'boroname', 'n_listings_2022', 'rate_2022', 'lisa_fdr_2022', 'lisa_p05_2022', 'lisa_pval_2022', 'status_2022', 'n_listings_2023', 'rate_2023', 'lisa_fdr_2023', 'lisa_p05_2023', 'lisa_pval_2023', 'status_2023', 'n_listings_2024', 'rate_2024', 'lisa_fdr_2024', 'lisa_p05_2024', 'lisa_pval_2024', 'status_2024', 'n_listings_2025', 'rate_2025', 'lisa_fdr_2025', 'lisa_p05_2025', 'lisa_pval_2025', 'status_2025', 'pct_change_22_25', 'transition']
